# Peningkatan LightGBM native
Jalankan dari root proyek. Tiga kandidat kapasitas dipilih pada fold 0 bersama bobot keputusan dan blend 50%/100%. Empat fold lain dipakai untuk konfirmasi dengan setelan dibekukan. Model leaderboard 0.59005 dipertahankan. Tidak ada neural network.


In [ ]:
import gc
import hashlib
import json
from pathlib import Path
import sys
project_root = Path.cwd() if (Path.cwd()/"scripts").is_dir() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from scripts.preprocess import ROOT, load_training
from scripts.train import build_pipeline, optimize_decisions
from scripts.predict import FoldPipelineEnsemble

CANDIDATES = [
    {'name':'compact', 'model__n_estimators':1000, 'model__num_leaves':63,
     'model__min_child_samples':60, 'model__reg_lambda':5},
    {'name':'long', 'model__n_estimators':1000, 'model__num_leaves':127,
     'model__min_child_samples':80, 'model__reg_lambda':5},
    {'name':'wide', 'model__n_estimators':700, 'model__num_leaves':255,
     'model__min_child_samples':100, 'model__reg_lambda':5},
]

def fit_candidate(config, fold, training, validation, X, y):
    """Fit or restore a complete pipeline with fold-local class weights and vocabulary."""
    path=ROOT/'models'/f"improved_{config['name']}_fold_{fold}.pkl"
    if path.exists():
        pipeline=joblib.load(path)
        print(f'Loaded {path.name}',flush=True)
    else:
        pipeline=build_pipeline('native_lightgbm',use_id_features=True)
        pipeline.set_params(**{key:value for key,value in config.items() if key!='name'})
        counts=np.bincount(y[training],minlength=6)[1:]
        sample_weight=np.sqrt(counts.mean()/counts)[y[training]-1]
        print(f"Training {config['name']} fold {fold+1}/5",flush=True)
        pipeline.fit(X.iloc[training],y[training],model__sample_weight=sample_weight)
        joblib.dump(pipeline,path)
    return pipeline


def run_improvement():
    """Select capacity/blend on fold 0, freeze settings, then evaluate folds 1–4."""
    fingerprint_paths=['data/train.csv','data/test.csv','data/sample_submission.csv',
        'scripts/train.py','scripts/preprocess.py','scripts/feature_engineering.py',
        'scripts/predict.py','notebooks/04_improve.ipynb','reports/native_predictions.npz']
    manifest={'candidates':CANDIDATES,'sha256':{}}
    for relative in fingerprint_paths:
        with (ROOT/relative).open('rb') as handle:
            manifest['sha256'][relative]=hashlib.file_digest(handle,'sha256').hexdigest()
    manifest_path=ROOT/'models/improved_manifest.json'
    if manifest_path.exists() and json.loads(manifest_path.read_text())!=manifest:
        raise ValueError('Improvement inputs changed; preserve/move improved_* checkpoints before a new experiment.')
    manifest_path.write_text(json.dumps(manifest,indent=2))
    X,ys=load_training(); y=ys.to_numpy()
    test=pd.read_csv(ROOT/'data/test.csv')
    baseline=np.load(ROOT/'reports/native_predictions.npz')
    splits=list(StratifiedKFold(5,shuffle=True,random_state=42).split(X,y))
    training,validation=splits[0]
    base_valid=baseline['oof'][validation]
    base_weights=baseline['decision_weights']
    trials=[{'name':'baseline','alpha':0.0,'weights':base_weights.tolist(),
        'selection_macro_f1':f1_score(y[validation],(base_valid*base_weights).argmax(1)+1,average='macro')}]
    for config in CANDIDATES:
        valid_path=ROOT/'reports'/f"improved_{config['name']}_selection.npz"
        if valid_path.exists():
            probability=np.load(valid_path)['validation']
        else:
            pipeline=fit_candidate(config,0,training,validation,X,y)
            probability=pipeline.predict_proba(X.iloc[validation])
            np.savez_compressed(valid_path,validation=probability)
            del pipeline; gc.collect()
        for alpha in [0.5,1.0]:
            blended=(1-alpha)*base_valid+alpha*probability
            weights=optimize_decisions(blended,y[validation])
            score=f1_score(y[validation],(blended*weights).argmax(1)+1,average='macro')
            trials.append({'name':config['name'],'alpha':alpha,'weights':weights.tolist(),'selection_macro_f1':score})
            print(f"Selection {config['name']} alpha={alpha}: {score:.6f}",flush=True)
        (ROOT/'reports/improved_selection.json').write_text(json.dumps(trials,indent=2))
    best=max(trials,key=lambda row:row['selection_macro_f1'])
    (ROOT/'reports/improved_frozen_selection.json').write_text(json.dumps(best,indent=2))
    print('Frozen selection: '+json.dumps(best),flush=True)
    if best['name']=='baseline':
        print('No candidate exceeded baseline on selection fold. Keep native submission.',flush=True)
        return best
    config=next(c for c in CANDIDATES if c['name']==best['name'])
    alpha=best['alpha']; weights=np.array(best['weights'])
    oof=np.zeros_like(baseline['oof']); test_probability=np.zeros_like(baseline['test'])
    fold_rows=[]
    for fold,(training,validation) in enumerate(splits):
        probability_path=ROOT/'reports'/f"improved_{config['name']}_fold_{fold}.npz"
        if probability_path.exists():
            checkpoint=np.load(probability_path)
            valid_probability,test_fold=checkpoint['validation'],checkpoint['test']
        else:
            pipeline=fit_candidate(config,fold,training,validation,X,y)
            print(f'Predicting selected candidate fold {fold+1}/5',flush=True)
            valid_probability=pipeline.predict_proba(X.iloc[validation])
            test_fold=pipeline.predict_proba(test)
            np.savez_compressed(probability_path,validation=valid_probability,test=test_fold)
            del pipeline;gc.collect()
        oof[validation]=(1-alpha)*baseline['oof'][validation]+alpha*valid_probability
        test_probability+=alpha*test_fold/5
        score=f1_score(y[validation],(oof[validation]*weights).argmax(1)+1,average='macro')
        fold_rows.append({'fold':fold,'role':'selection' if fold==0 else 'confirmation','macro_f1':score})
        pd.DataFrame(fold_rows).to_csv(ROOT/'reports/improved_fold_scores.csv',index=False)
        print(f'Improved fold {fold+1}/5: {score:.6f}',flush=True)
    test_probability+=(1-alpha)*baseline['test']
    prediction=(oof*weights).argmax(1)+1
    confirmation=np.concatenate([v for _,v in splits[1:]])
    base_prediction=(baseline['oof']*base_weights).argmax(1)+1
    report={'selection':best,'config':config,'cv_macro_f1':f1_score(y,prediction,average='macro'),
        'confirmation_macro_f1':f1_score(y[confirmation],prediction[confirmation],average='macro'),
        'baseline_confirmation_macro_f1':f1_score(y[confirmation],base_prediction[confirmation],average='macro'),
        'baseline_cv_macro_f1':f1_score(y,base_prediction,average='macro'),
        'classification_report':classification_report(y,prediction,output_dict=True),
        'protocol':'Capacity, blend and decision weights selected on fold 0 only. Frozen for folds 1-4. These folds have historical reuse; this is not a fresh unbiased benchmark.',
        'baseline_leaderboard_user_reported':0.59005,'new_leaderboard':None}
    report['confirmation_improved']=report['confirmation_macro_f1']>report['baseline_confirmation_macro_f1']
    (ROOT/'reports/improved_model_metrics.json').write_text(json.dumps(report,indent=2))
    np.savez_compressed(ROOT/'reports/improved_predictions.npz',oof=oof,test=test_probability,y=y,decision_weights=weights)
    filenames=[f"improved_{config['name']}_fold_{fold}.pkl" for fold in range(5)]
    if alpha==0.5:
        filenames += [f'native_fold_{fold}.pkl' for fold in range(5)]
    ensemble=FoldPipelineEnsemble(filenames,weights.tolist())
    joblib.dump(ensemble,ROOT/'models/improved_ensemble.pkl')
    sample=pd.read_csv(ROOT/'data/sample_submission.csv')
    assert sample.building_id.is_unique and test.building_id.is_unique
    assert set(sample.building_id)==set(test.building_id)
    mapping=pd.Series((test_probability*weights).argmax(1)+1,index=test.building_id)
    sample['damage_grade']=sample.building_id.map(mapping)
    assert not sample.isna().any().any() and sample.damage_grade.isin(range(1,6)).all()
    sample.damage_grade=sample.damage_grade.astype('int64')
    sample.to_csv(ROOT/'submissions/submission_improved.csv',index=False)
    print(json.dumps(report,indent=2),flush=True)
    return report

result=run_improvement()
result
